# Results Analysis

Load the latest evaluation record for each model from `results/`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from utils import TFLiteEvalRecord

repo_root = Path.cwd().parent
results_dir = repo_root / "results"

records = []
for p in sorted(results_dir.glob("*.jsonl")):
    with p.open("r", encoding="utf-8") as f:
        lines = f.readlines()
    if not lines:
        continue
    last = json.loads(lines[-1])
    records.append(TFLiteEvalRecord.model_validate(last))

records = sorted(records, key=lambda r: r.model_name)
print(f"Loaded {len(records)} runs from {results_dir}")


ROC curves (computed on train set).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

for rec in records:
    if rec.train.roc_fpr is None or rec.train.roc_tpr is None or rec.train.auc is None:
        continue
    label = f"{rec.model_name} (AUC={rec.train.auc:.2f})"
    for ax in axes:
        ax.plot(rec.train.roc_fpr, rec.train.roc_tpr, label=label)

for ax, xscale, title in zip(axes, ["log", "linear"], ["ROC (log scale)", "ROC (linear scale)"]):
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
    ax.set_xscale(xscale)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), fontsize=9)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)

plt.tight_layout()
plt.subplots_adjust(bottom=0.28)
plt.show()


# Table IV — threshold and prediction metrics on **training set**.

In [ ]:
rows = []
for rec in records:
    m = rec.train
    rows.append(
        {
            "Model": rec.model_name,
            "Threshold (t)": m.threshold,
            "Acc.": m.accuracy,
            "Precision": m.precision,
            "Recall": m.recall,
            "F2": m.f2,
        }
    )

df_train = pd.DataFrame(rows).sort_values("F2",ascending=False)

df_train.style.format(
    {
        "Threshold (t)": "{:.3e}",
        "Acc.": "{:.3f}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "F2": "{:.3f}",
    }
)


# Table V — prediction metrics on **test set**.

In [ ]:
rows = []
for rec in records:
    m = rec.test
    rows.append(
        {
            "Model": rec.model_name,
            "Accuracy": m.accuracy,
            "Precision": m.precision,
            "Recall": m.recall,
            "F2": m.f2,
            "Avg Inference (ms)": m.avg_inference_time_ms if m.avg_inference_time_ms is not None else np.nan,
        }
    )

pd.DataFrame(rows).sort_values("Model").style.format(
    {
        "Accuracy": "{:.2f}",
        "Precision": "{:.2f}",
        "Recall": "{:.2f}",
        "F2": "{:.2f}",
        "Avg Inference (ms)": "{:.3f}",
    },
    na_rep="—",
)
